## 准备数据

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [8]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [2]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        ####################
        self.W1 = tf.Variable(tf.random.normal([784, 128], stddev=0.1))
        self.b1 = tf.Variable(tf.zeros([128]))
        self.W2 = tf.Variable(tf.random.normal([128, 10], stddev=0.1))
        self.b2 = tf.Variable(tf.zeros([10]))

    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        ####################
        x = tf.reshape(x, [-1, 784])
        h = tf.nn.relu(x @ self.W1 + self.b1)
        logits = h @ self.W2 + self.b2
        return logits
        
model = myModel()

optimizer = optimizers.Adam()


## 计算 loss

In [4]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [5]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 2.5060453 ; accuracy 0.086733334
epoch 1 : loss 2.4908102 ; accuracy 0.088583335
epoch 2 : loss 2.4760683 ; accuracy 0.09136666
epoch 3 : loss 2.4617813 ; accuracy 0.094066665
epoch 4 : loss 2.4479144 ; accuracy 0.09728333
epoch 5 : loss 2.4344404 ; accuracy 0.10053334
epoch 6 : loss 2.421332 ; accuracy 0.10405
epoch 7 : loss 2.4085598 ; accuracy 0.10755
epoch 8 : loss 2.3961003 ; accuracy 0.111666664
epoch 9 : loss 2.3839352 ; accuracy 0.116116665
epoch 10 : loss 2.3720465 ; accuracy 0.121316664
epoch 11 : loss 2.3604152 ; accuracy 0.12615
epoch 12 : loss 2.3490262 ; accuracy 0.131
epoch 13 : loss 2.337865 ; accuracy 0.1362
epoch 14 : loss 2.3269196 ; accuracy 0.14208333
epoch 15 : loss 2.3161774 ; accuracy 0.14776666
epoch 16 : loss 2.3056283 ; accuracy 0.15305
epoch 17 : loss 2.295263 ; accuracy 0.15928334
epoch 18 : loss 2.2850714 ; accuracy 0.16545
epoch 19 : loss 2.2750435 ; accuracy 0.17123333
epoch 20 : loss 2.2651708 ; accuracy 0.17751667
epoch 21 : loss 2.25544